In [26]:
import os
import pandas as pd
import numpy as np

# Create a new directory for storing exported csv files
path = os.getcwd()+'/csv'
isExist = os.path.exists(path)
if not isExist:
    os.makedirs(path)

# Get all sheet names from the original Excel for csv exporting
xls = pd.ExcelFile('mrtssales92-present.xls')
sheets = xls.sheet_names[1:]

# Import the Excel into DataFrames by sheet name, which involves adding headers, slicing rows and columns for use,
# and including a "Total" column for the adjusted values, which were not originally present 
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for sheet in sheets:
    col_labels = ['Kind of Business']
    for month in months:
        col_labels.append(month+' '+sheet)
    col_labels.append('Total')
    # Specify the columns and rows for use through the specific arguments of the read_excel function
    df_sheet = pd.read_excel('mrtssales92-present.xls', sheet_name=sheet, header=None, names=col_labels,
                          index_col=0, usecols=list(range(1,15)), skiprows=6, nrows=65)
    # Export the organized DataFrame of each sheet to a csv and save it in the pre-specified directory
    df_sheet.to_csv(f'{path}/{sheet}.csv')
    print(f'"{sheet}.csv" saved in folder "csv" successfully!')

# Sales data is only available for the first 2 months of 2021, it's necessary to differentiate its table structure
# Customize and export the sheet for 2021 separately through the following code
col_2021 = ['Kind of Business', 'Jan 2021', 'Feb 2021']
df_2021 = pd.read_excel('mrtssales92-present.xls', sheet_name='2021', header=None, names=col_2021,
                          index_col=0, usecols=list(range(1,4)), skiprows=6, nrows=65)
df_2021['Total'] = df_2021.sum(axis=1)
df_2021.to_csv(f'{path}/2021.csv')
print(f'"2021.csv" saved in folder "csv" successfully!')

"2020.csv" saved in folder "csv" successfully!
"2019.csv" saved in folder "csv" successfully!
"2018.csv" saved in folder "csv" successfully!
"2017.csv" saved in folder "csv" successfully!
"2016.csv" saved in folder "csv" successfully!
"2015.csv" saved in folder "csv" successfully!
"2014.csv" saved in folder "csv" successfully!
"2013.csv" saved in folder "csv" successfully!
"2012.csv" saved in folder "csv" successfully!
"2011.csv" saved in folder "csv" successfully!
"2010.csv" saved in folder "csv" successfully!
"2009.csv" saved in folder "csv" successfully!
"2008.csv" saved in folder "csv" successfully!
"2007.csv" saved in folder "csv" successfully!
"2006.csv" saved in folder "csv" successfully!
"2005.csv" saved in folder "csv" successfully!
"2004.csv" saved in folder "csv" successfully!
"2003.csv" saved in folder "csv" successfully!
"2002.csv" saved in folder "csv" successfully!
"2001.csv" saved in folder "csv" successfully!
"2000.csv" saved in folder "csv" successfully!
"1999.csv" sa

In [13]:
#!pip install mysql-connector-python
import mysql.connector
import yaml
import os
from sqlalchemy import create_engine
import pandas as pd

#doc = yaml.safe_load(open('mysql_cnx_config.yml'))
cnx = mysql.connector.connect(
    user= 'root',
    password='Root123',
    host='127.0.0.1',
    database= 'mrts',
    auth_plugin='mysql_native_password'
)

doc = yaml.safe_load(open('mysql_cnx_config.yml'))
config = {
    'user':doc['user'],
    'passwd':doc['passwd'],
    'host':doc['host'],
    }

# Create an SQLAlchemy engine for MySQL via the assigned driver 'MySQL Connectors'
# url should follow 'mysql+mysqlconnector://<user>:<password>@<host>[:<port>]/<dbname>' format
url = 'mysql+mysqlconnector://{0}:{1}@{2}'.format(
    config['user'],config['passwd'],config['host'])
engine = create_engine(url)

#try:
    # Install the driver 'MySQL Connectors'
    #cnx = mysql.connector.connect(**config)
cursor = cnx.cursor()

    # Create the database by executing SQL queries directly on the created engine object 
    #cursor.execute('CREATE DATABASE IF NOT EXISTS mrts')
    #print('Database "mrts" created successfully!')

    # Get the list of csv files from the directory in a defensive way
fi_ls = [fi for fi in os.listdir(os.getcwd()+'/csv') if fi.endswith('.csv')]

    # Store the looped csv files in DataFrames and write them as tables into the SQL database via the engine
for fi in fi_ls:
    df = pd.read_csv(os.path.join(os.getcwd(),'csv',fi))
        # Use to_sql(name,con,schema=None,if_exists='fail',index=True) method to load DataFrames into the database
        # The engine must be assigned to argument 'con'. The loading process does not success without SQLAlchemy
    tbl = fi.replace('.csv','')
    df.to_sql(f'{tbl}', con='engine', schema='mrts', if_exists='replace', index=False)
    print(f'Data from "{fi}" loaded into table "{tbl}" successfully!')

#except mysql.connector.Error as err:
    #print(f'Error: {err}')

ArgumentError: Could not parse rfc1738 URL from string 'engine'